# Практикум 6. kNN

Сопровождает интерактивную лабораторию №6 («Метод k ближайших соседей»). Тема курса: классификация методом k-ближайших соседей

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.5, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Подбор k по кросс-валидации

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split

vec = TfidfVectorizer()
X = vec.fit_transform(df.text); y = df.label.values
ks = range(1, 31)
sc = [cross_val_score(KNeighborsClassifier(k, metric="cosine", algorithm="brute"), X, y, cv=5).mean() for k in ks]
plt.plot(ks, sc, marker="o"); plt.xlabel("k"); plt.ylabel("accuracy (CV)"); plt.title("kNN: точность и число соседей"); plt.show()
print("Лучшее k:", ks[int(np.argmax(sc))], round(max(sc), 3))

## 2. Метрика и взвешивание

In [ ]:
rows = []
for metric in ["cosine", "euclidean"]:
    for w in ["uniform", "distance"]:
        m = KNeighborsClassifier(7, metric=metric, weights=w, algorithm="brute")
        rows.append((metric, w, cross_val_score(m, X, y, cv=5).mean()))
pd.DataFrame(rows, columns=["метрика", "голоса", "accuracy"]).round(3)

## 3. Кто соседи: объяснимость kNN

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(df.text, df.label, test_size=0.3, random_state=0)
vec2 = TfidfVectorizer().fit(Xtr)
knn = KNeighborsClassifier(5, metric="cosine", algorithm="brute").fit(vec2.transform(Xtr), ytr)
query = "команда гол матч бюджет закон"
dist, idx = knn.kneighbors(vec2.transform([query]), 5)
print("Запрос:", query, "->", knn.predict(vec2.transform([query]))[0])
for d, i in zip(dist[0], idx[0]):
    print(f"сходство {1-d:.2f} | {ytr.iloc[i]:11s} | {Xtr.iloc[i]}")

## Задания

1. Постройте график точности по k на обучающих данных (без кросс-валидации). Почему при k=1 точность 100%?
2. Сравните время предсказания kNN и логистической регрессии при росте корпуса (n_per_class=1500). Что это говорит об отсутствии «обучения»?
3. Попробуйте TF (CountVectorizer) вместо TF-IDF. Как меняется точность?